# Problems & Goal

- Bài toán cốt lõi: Phân loại đơn nhãn (single-label) 6 lớp theo đúng benchmark. Mặc dù thực tế các lớp có sự chồng chéo ngữ nghĩa (ví dụ: ảnh Tết thường bao gồm cảnh tụ họp), ta sẽ xử lý sự mơ hồ này trực tiếp ở khâu huấn luyện và phân tích lỗi thay vì thay đổi định nghĩa bài toán.
- Vai trò của lớp "other": Hoạt động như một phễu lọc (catch-all failure mode) để hứng các mẫu ngoại lai hoặc không rõ ràng, chứ không mang một đặc trưng ngữ nghĩa độc lập.
- Thách thức từ dữ liệu: Kích thước tập mẫu cực nhỏ, mất cân bằng nghiêm trọng, nhãn nhiễu và ranh giới phân loại mờ nhạt (ví dụ: ảnh công viên dễ nhầm thành thiên nhiên).
- Tiêu chí tối ưu: Ưu tiên tính mạnh mẽ (robustness) và độ tin cậy khi đưa vào thực tế. Không chạy đua tối ưu độ chính xác điểm (point accuracy) trên tập dữ liệu nhỏ vốn rất dễ bị overfit.
- Chiến lược thực thi: Tiếp cận theo hướng tối ưu dữ liệu (data-centric) thay vì dùng mạng end-to-end phức tạp. Giải pháp là sử dụng Frozen Embeddings (SigLIP2 đa ngôn ngữ) kết hợp Classification Head có trọng số, đi kèm kỹ thuật hiệu chuẩn (calibration) và lọc nhiễu offline (CLIPCleaner).

# EDA

## Manifest

In [ ]:
from pathlib import Path

from z_photos.conf import Config

cfg = Config(
    data_root=Path("../data"),
    seed=42,
)

## Bronze: FiftyOne Dataset

TODO(Explain why FiftyOne)

In [ ]:
import z_photos
from z_photos.datasets import build_dataset

dataset = build_dataset(z_photos.__name__, bronze_dir=cfg.bronze_dir)
dataset

 100% |█████████████████| 261/261 [41.6ms elapsed, 0s remaining, 6.3K samples/s]   
 100% |███████████████████| 60/60 [8.1ms elapsed, 0s remaining, 7.4K samples/s]       
Computing metadata...
 100% |█████████████████| 321/321 [131.9ms elapsed, 0s remaining, 2.4K samples/s]    


Name:        z_photos
Media type:  image
Num samples: 321
Persistent:  True
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Classification)
    raw_label:        fiftyone.core.fields.StringField

## Inventory & Sanity

1. Class distribution: mất cân bằng giữa train/test?
1. Metadata sanity: kích thước ảnh, file corrupt?
1. Exact duplicates: file trùng lặp chính xác (hash).

In [ ]:
from collections import Counter

import numpy as np
import pandas as pd

# Class distribution per split
train_view = dataset.match_tags("train")
test_view = dataset.match_tags("test")

train_counts = Counter(train_view.values("ground_truth.label"))
test_counts = Counter(test_view.values("ground_truth.label"))

all_classes = sorted(set(train_counts) | set(test_counts))
df_dist = pd.DataFrame(
    {
        "class": all_classes,
        "train": [train_counts.get(c, 0) for c in all_classes],
        "test": [test_counts.get(c, 0) for c in all_classes],
    }
).set_index("class")
df_dist["total"] = df_dist["train"] + df_dist["test"]
df_dist["train_pct"] = (df_dist["train"] / df_dist["train"].sum() * 100).round(1)
df_dist["test_pct"] = (df_dist["test"] / df_dist["test"].sum() * 100).round(1)

print("= Class Distribution")
print(f"Train total: {df_dist['train'].sum()}")
print(f"Test total: {df_dist['test'].sum()}")
df_dist.index.name = None
df_dist.style.format({"train_pct": "{:.1f}%", "test_pct": "{:.1f}%"})

= Class Distribution
Train total: 261
Test total: 60


,train,test,total,train_pct,test_pct
baby_playing,39,9,48,14.9%,15.0%
gathering,24,5,29,9.2%,8.3%
lunar_new_year,32,7,39,12.3%,11.7%
nature,60,14,74,23.0%,23.3%
other,66,16,82,25.3%,26.7%
trekking,40,9,49,15.3%,15.0%


In [ ]:
widths = dataset.values("metadata.width")
heights = dataset.values("metadata.height")
sizes = dataset.values("metadata.size_bytes")

missing_meta = sum(1 for w in widths if w is None)
print(f"Samples với metadata thiếu (có thể corrupt): {missing_meta}")

w_arr = np.array([w for w in widths if w], dtype=float)
h_arr = np.array([h for h in heights if h], dtype=float)
s_arr = np.array([s for s in sizes if s], dtype=float)
ar_arr = w_arr / h_arr

df_meta = pd.DataFrame(
    {
        "metric": ["Width (px)", "Height (px)", "Aspect ratio", "File size (KB)"],
        "min": [w_arr.min(), h_arr.min(), ar_arr.min(), s_arr.min() / 1024],
        "max": [w_arr.max(), h_arr.max(), ar_arr.max(), s_arr.max() / 1024],
        "mean": [w_arr.mean(), h_arr.mean(), ar_arr.mean(), s_arr.mean() / 1024],
        "std": [w_arr.std(), h_arr.std(), ar_arr.std(), s_arr.std() / 1024],
    }
).set_index("metric")
df_meta.index.name = None
df_meta.style.format("{:.1f}")

Samples với metadata thiếu (có thể corrupt): 0


,min,max,mean,std
Width (px),194.0,3464.0,989.5,483.3
Height (px),168.0,3072.0,694.9,379.0
Aspect ratio,0.7,3.2,1.5,0.3
File size (KB),9.6,4212.4,314.6,490.3


## Exact Duplicates

`fob.compute_exact_duplicates`: so sánh hash file MD5/SHA, phát hiện file trùng chính xác. Không cần model.

In [ ]:
import fiftyone.brain as fob
import pandas as pd

exact_dups = fob.compute_exact_duplicates(dataset, progress=False)

rows = []
for rep_id, dup_ids in list(exact_dups.items())[:5]:
    rep = dataset[rep_id]
    for did in dup_ids:
        s = dataset[did]
        rows.append(
            {
                "Rep Label": rep.ground_truth.label,
                "Rep File": rep.filepath.split("/")[-1],
                "Dup Label": s.ground_truth.label,
                "Dup File": s.filepath.split("/")[-1],
            }
        )

pd.DataFrame(rows)

Computing filehashes...


,Rep Label,Rep File,Dup Label,Dup File
0,baby_playing,0036_f4a414c4155d.png,other,0036_f4a414c4155d.png
1,lunar_new_year,0008_8843ddf7b700.jpg,other,tet_0008_8843ddf7b700.jpg
2,lunar_new_year,0029_5402f43a6653.jpg,other,tet_0029_5402f43a6653.jpg
3,lunar_new_year,0034_a077e240af5b.jpg,other,tet_0034_a077e240af5b.jpg
4,lunar_new_year,0036_adb92c35a73c.jpg,other,tet_0036_adb92c35a73c.jpg


## SigLIP2: Semantic Index


In [ ]:
import torch

from z_photos.zoo import SigLIP2, SigLIP2Config

siglip2_fo = SigLIP2(
    SigLIP2Config(
        {
            "model_path": "google/siglip2-so400m-patch16-384",
            "dtype": torch.float16,
        }
    )
)
dataset.compute_embeddings(model=siglip2_fo, embeddings_field="siglip2_embeddings")

Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

 100% |█████████████████| 321/321 [1.2m elapsed, 0s remaining, 6.5 samples/s]      


In [ ]:
import fiftyone.brain

fiftyone.brain.brain_config.similarity_backends.keys()

dict_keys(['sklearn', 'pinecone', 'qdrant', 'milvus', 'lancedb', 'redis', 'mongodb', 'elasticsearch', 'pgvector', 'mosaic'])

In [ ]:
fob.compute_visualization(
    dataset, embeddings="siglip2_embeddings", brain_key="siglip2_viz"
)
print("SigLIP2 visualization OK.")

siglip2_sim = fob.compute_similarity(
    dataset,
    model=siglip2_fo,
    embeddings="siglip2_embeddings",
    brain_key="siglip2_sim",
    backend="qdrant",
)
print(f"SigLIP2 similarity index ok. supports_prompts={siglip2_sim.supports_prompts}")

Generating visualization...
UMAP( verbose=True)
Fri Mar 13 20:40:51 2026 Construct fuzzy simplicial set
Fri Mar 13 20:40:51 2026 Finding Nearest Neighbors
Fri Mar 13 20:40:51 2026 Finished Nearest Neighbor Search
Fri Mar 13 20:40:51 2026 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Fri Mar 13 20:40:52 2026 Finished embedding
SigLIP2 visualization OK.
This index will not support prompt queries in the App or in future Python sessions. You can support this by providing the string name of a zoo model rather than a Model instance to compute_similarity(model=).


ModuleNotFoundError: No module named 'qdrant_client'